# IndoBERT Pantun Core Pipeline (Colab)

Versi ini adalah **core pipeline** tanpa patch eksperimen.

Alur:
1. IndoBERT token labeling
2. Routing label -> jalur fonemisasi
3. CRF BI untuk pemenggalan suku kata dari fonem
4. Ekstraksi rima akhir + output ringkas pantun


In [ ]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk sklearn-crfsuite joblib pyphen

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# UBAH sesuai lokasi Anda
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi.joblib'


In [ ]:
import os
import re
import joblib
import torch
import nltk
import pyphen
from functools import lru_cache
from transformers import AutoTokenizer, AutoModelForTokenClassification
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('tokenizers/punkt', 'punkt')
_ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()
id2label = model.config.id2label
print('Labels:', id2label)

crf_bi_model = None
if CRF_BI_MODEL_PATH and os.path.exists(CRF_BI_MODEL_PATH):
    crf_bi_model = joblib.load(CRF_BI_MODEL_PATH)
    print('[CRF3] Loaded:', CRF_BI_MODEL_PATH)
else:
    print('[CRF3] Not found:', CRF_BI_MODEL_PATH)

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()
pyphen_en_dict = pyphen.Pyphen(lang='en_US')


In [ ]:
LABELS_ASING = {'B-ASG', 'I-ASG'}
LABELS_SINGKATAN = {'B-SKT', 'I-SKT'}
LABELS_DISEMVOWEL = {'B-DSM', 'I-DSM'}
LABELS_TYPO = {'B-TYP', 'I-TYP'}
LABELS_VALID = {'O', *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}

VOWELS = set('aiueo')
COMMON_DIGRAPHS = ('ng', 'ny', 'kh', 'sy', 'br', 'tr', 'kr', 'pr', 'gr', 'dr', 'st', 'sk')
CRF3_VOWELS = {'a', 'i', 'u', 'e', 'o', 'ə', 'ɛ', 'ɪ', 'ʊ', 'ɔ', 'æ', 'ɐ', 'ɒ', 'ɑ', 'ʌ', 'ɝ'}
CRF3_DIPHTHONGS = ('eɪ', 'aɪ', 'aʊ', 'ɔɪ', 'oʊ')

ALPHABET_EJA_MAP = {
    'a': 'a', 'b': 'be', 'c': 'ce', 'd': 'de', 'e': 'e', 'f': 'ef', 'g': 'ge',
    'h': 'ha', 'i': 'i', 'j': 'je', 'k': 'ka', 'l': 'el', 'm': 'em', 'n': 'en',
    'o': 'o', 'p': 'pe', 'q': 'ki', 'r': 'er', 's': 'es', 't': 'te', 'u': 'u',
    'v': 've', 'w': 'we', 'x': 'eks', 'y': 'ye', 'z': 'zet'
}

ARPABET_TO_IPA = {
    'AA': 'a', 'AE': 'æ', 'AH': 'ə', 'AO': 'ɔ', 'AW': 'aʊ', 'AY': 'aɪ',
    'B': 'b', 'CH': 'tʃ', 'D': 'd', 'DH': 'ð', 'EH': 'ɛ', 'ER': 'ɝ', 'EY': 'eɪ',
    'F': 'f', 'G': 'g', 'HH': 'h', 'IH': 'ɪ', 'IY': 'i', 'JH': 'dʒ', 'K': 'k',
    'L': 'l', 'M': 'm', 'N': 'n', 'NG': 'ŋ', 'OW': 'oʊ', 'OY': 'ɔɪ', 'P': 'p',
    'R': 'r', 'S': 's', 'SH': 'ʃ', 'T': 't', 'TH': 'θ', 'UH': 'ʊ', 'UW': 'u',
    'V': 'v', 'W': 'w', 'Y': 'j', 'Z': 'z', 'ZH': 'ʒ'
}


def normalize_token(token: str) -> str:
    t = token.lower().strip()
    t = ''.join(ch for ch in t if ch.isalpha())
    return re.sub(r'(.)\1{2,}', r'\1\1', t)


def route_by_label(label: str) -> str:
    norm = str(label).upper().strip()
    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def predict_token_labels(text: str):
    words = text.strip().split()
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', truncation=True, padding=False)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()
    rows = []
    cur = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != cur:
            cur = widx
            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            conf = float(top2.values[0].item())
            second = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            label = id2label[pred_ids[idx]]
            rows.append({
                'token': words[widx],
                'label': label,
                'confidence': conf,
                'margin': conf - second,
                'route': route_by_label(label),
            })
    return rows


In [ ]:
def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return ' '.join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ''
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(' ', '')


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r'\d', '', token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return ''.join(ipa_tokens)


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ''
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != ' ' and p != '<unk>']
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def singkatan_to_ejaan(word: str) -> str:
    clean = ''.join(ch for ch in word.lower() if ch.isalpha())
    return ' '.join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def rumus_a_singkatan_simple(word: str) -> str:
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ''
    return g2p_id_phoneme(ejaan)

def process_disemvowelling_phonemes(word: str) -> str:
    """
    Rumus B/C: Untuk kata disemvowelling dan typo
    """
    norm = normalize_token(word)
    if not norm:
        return ''

    # Jika mengandung vokal, jalankan g2p_id langsung
    if any(ch in VOWELS for ch in norm):
        return g2p_id_phoneme(norm)

    # Heuristik kluster konsonan (misal: 'brgkt' -> 'berangkat' / 'b-e-r-a-n-g-k-a-t')
    # Menyisipkan vokal pembantu pada urutan konsonan
    reconstructed = []
    for i, ch in enumerate(norm):
        reconstructed.append(ch)
        if i < len(norm) - 1 and norm[i+1] not in VOWELS:
            # Sisipkan e-pepet jika di antara konsonan
            if ch not in ('g', 'n', 'y', 'k', 's'):
                reconstructed.append('e')
    
    candidate = ''.join(reconstructed)
    return g2p_id_phoneme(candidate)


def phonematize_token(token: str, route: str) -> str:
    """
    Eksekusi fonemisasi berdasarkan rute dari Layer 2 Decision Logic.
    """
    norm = normalize_token(token)
    if not norm:
        return ''
    
    if route == 'RUMUS_A_FONEMISASI_ALFABETIS':
        # Kondisi A: Singkatan / PROPN (e.g. dpr -> de-pe-er)
        return rumus_a_singkatan_simple(norm)
    
    elif route == 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING':
        # Kondisi B: Disemvowelling / Typo (e.g. brgkt)
        return process_disemvowelling_phonemes(norm)
    
    elif route == 'FAILSAFE_G2P_ASING':
        # Kondisi C: Bahasa Asing / Tag X (e.g. healing)
        return g2p_foreign_phoneme(norm)
    
    else:
        # Kondisi D: Kata Baku IDN (CRF1_G2P_KATA_BAKU)
        return g2p_id_phoneme(norm)

def get_phonemes_for_sentence(tagged_tokens: list[dict]) -> str:
    """
    Menggabungkan seluruh token terproses menjadi Rangkaian Fonem Bersih (IPA).
    """
    phoneme_list = []
    for item in tagged_tokens:
        ph = phonematize_token(item['token'], item['route'])
        if ph:
            phoneme_list.append(ph)
    return ' '.join(phoneme_list)

In [ ]:
def extract_phoneme_features(phoneme_seq: str) -> list[dict]:
    """
    Mengekstrak fitur biner untuk setiap karakter fonem untuk input CRF 2 Syllabifier.
    """
    features = []
    for i, ch in enumerate(phoneme_seq):
        is_v = ch.lower() in CRF3_VOWELS
        feat = {
            'bias': 1.0,
            'char': ch,
            'is_vowel': is_v,
            'is_consonant': not is_v and ch.isalpha(),
            'is_first': i == 0,
            'is_last': i == len(phoneme_seq) - 1,
            'prev_char': '' if i == 0 else phoneme_seq[i-1],
            'next_char': '' if i == len(phoneme_seq) - 1 else phoneme_seq[i+1],
        }
        features.append(feat)
    return features

def syllabify_phonemes(phoneme_str: str) -> list[str]:
    """
    Memotong sekuens fonem menjadi suku kata menggunakan Cascaded CRF 2.
    """
    clean_ph = phoneme_str.replace(' ', '')
    if not clean_ph:
        return []
    
    features = extract_phoneme_features(clean_ph)
    preds = crf_bi_model.predict([features])[0]
    
    syllables = []
    curr = ""
    for char, label in zip(clean_ph, preds):
        if label == 'B' and curr:
            syllables.append(curr)
            curr = char
        else:
            curr += char
    if curr:
        syllables.append(curr)
    return syllables

In [ ]:
def extract_end_rhyme(syllables: list[str]) -> str:
    """
    Mengekstrak bunyi rima akhir dari suku kata terakhir larik pantun.
    """
    if not syllables:
        return ''
    
    last_syllable = syllables[-1]
    # Mengambil vokal terakhir hingga akhir suku kata
    vowel_idx = -1
    for i, ch in enumerate(last_syllable):
        if ch.lower() in CRF3_VOWELS:
            vowel_idx = i
            break
    
    if vowel_idx != -1:
        return last_syllable[vowel_idx:]
    return last_syllable


def process_pantun_line(line_text: str) -> dict:
    """
    Pemrosesan end-to-end 1 Larik Pantun melalui Layer 1 -> Layer 2 -> Layer 3.
    """
    # 1. Layer 1: Token Labeling & Decision Switch
    tagged_tokens = predict_token_labels(line_text)
    
    # 2. Layer 2 & 2b: Switchboard Fonemisasi -> Rangkaian Fonem IPA
    phonemes = get_phonemes_for_sentence(tagged_tokens)
    
    # 3. Layer 3: CRF BI Syllabifier -> Pemenggalan Suku Kata
    syllables = syllabify_phonemes(phonemes)
    
    # 4. Output Metrum & Rima
    metrum_count = len(syllables)
    rima_akhir = extract_end_rhyme(syllables)
    
    return {
        'text': line_text,
        'tagged_tokens': tagged_tokens,
        'phonemes': phonemes,
        'syllables': syllables,
        'metrum_count': metrum_count,
        'rima_akhir': rima_akhir
    }

In [ ]:
def analyze_pantun(pantun_text: str):
    """
    Menjalankan alur penuh IndoBERT Pantun Core Pipeline pada 4 larik pantun.
    """
    lines = [line.strip() for line in pantun_text.strip().split('\n') if line.strip()]
    
    print("=" * 60)
    print("      INDOBERT PANTUN CORE PIPELINE - RESULT ANALYSES")
    print("=" * 60)
    
    results = []
    rima_pattern = []
    
    for idx, line in enumerate(lines, 1):
        res = process_pantun_line(line)
        results.append(res)
        rima_pattern.append(res['rima_akhir'])
        
        print(f"\n[Larik {idx}]: \"{res['text']}\"")
        print(f"├─ Fonem Bersih (IPA) : /{res['phonemes']}/")
        print(f"├─ Suku Kata (CRF BI) : {' - '.join(res['syllables'])}")
        print(f"├─ Metrum (Total Syll): {res['metrum_count']} suku kata")
        print(f"└─ Rima Akhir         : /{res['rima_akhir']}/")
    
    print("\n" + "-" * 60)
    print("SUMMARY PANTUN:")
    print(f"• Total Larik  : {len(results)}")
    print(f"• Skema Metrum : {[r['metrum_count'] for r in results]} suku kata")
    print(f"• Pola Rima    : {' - '.join(rima_pattern)}")
    print("-" * 60)

# -------------------------------------------------------------
# CONTOH PENGUJIAN PIPELINE
# -------------------------------------------------------------
sample_pantun = """
saya otw ke dpr mau healing blm brgkt
pergi belanja membeli kain
tetap tersenyum walau berat
agar hidup menjadi tenang
"""

# Eksekusi Analisis
analyze_pantun(sample_pantun)